In [7]:
%pip install pydantic[email]

Note: you may need to restart the kernel to use updated packages.


In [28]:
#Ficou claro que o pydantic é muito útil para validar e serializar dados, por isso foi escolhido dois campos para fazer a demosntracão de conhecimento 
import enum
import hashlib
import re
from typing import Any, Self

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    field_serializer,
    model_serializer,
    SecretStr,
)

#definindo a validade dos dados, o que é necessário ter e quantos caracteres são necessários
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")  #letra maiuscula, minuscula e número, 8 caracteres (exemplo)
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")
VALID_CPF_REGEX = re.compile(r"^\d{11}$")
VALID_ADDRESS_REGEX = re.compile(r"^.{7,}$")

#Apenas dois tipos de usuário para facilitar
class Role(enum.IntEnum):
    User = 1
    Admin = 2

#definindo quais campos o usuário terá, ou seja, quais serão seus dados
class User(BaseModel):
    name: str = Field(examples=["Arjan"])

    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )

    password: SecretStr = Field(
        examples=["Password123"],
        description="The password of the user",
        exclude=True,
    )

    role: Role = Field(
        description="The role of the user",
        default=Role.User,
        validate_default=True,
    )

    cpf: str = Field(
        description="The CPF of the user",
        frozen=True
    )

    address: str = Field(
        examples=["Rua abc Qd. 01 Lt 01 - 00000-000"],
        description="The address of the user",
        frozen=True
    )

#Validacão do campo nome, ele irá rodar quando o campo nome for lido
    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

#Validacão do campo CPF, ele irá rodar quando o campo CPF for lido
    @field_validator("cpf")
    @classmethod
    def validate_cpf(cls, v: str) -> str:
        if not VALID_CPF_REGEX.match(v):
            raise ValueError(
                "CPF is invalid, must contain exactly 11 numbers"
            )
        return v

#Validacão do campo endereco, ele irá rodar quando o campo adress for lido
    @field_validator("address")
    @classmethod
    def validate_address(cls, v: str) -> str:
        if not VALID_ADDRESS_REGEX.match(v):
            raise ValueError(
                "Address is invalid, must contain at least 5 characters"
            )
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

#Validacao de todos os campos juntos, verifica se senha tem o nome também, valida a senha e criptografa
    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:

        required_fields = ["name", "password", "cpf", "address"]

        for field in required_fields:
            if field not in v:
                raise ValueError(f"{field} is required")

        password = v["password"]

        if v["name"].casefold() in password.casefold():
            raise ValueError("Password cannot contain name")

        if not VALID_PASSWORD_REGEX.match(password):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )

#Criptografia
        v["password"] = hashlib.sha256(password.encode()).hexdigest()

        return v

#Apenas o Arjan será admin, como se fosse um sistema real
    @model_validator(mode="after")
    def validate_user_post(self) -> Self:
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

#Adicionado os campos para o good_data e feito os bad_data para cada campo adicional (cpf e address)
def main() -> None:
    test_data = dict(
        good_data = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
            "cpf": "111.111.111-11",
            "adress": "Rua AB 123",
        },
        bad_role = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "bad password",
        },
        bad_cpf = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "password123",
            "cpf": "111.11.111-1",
        },
        bad_adress = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "password123",
            "cpf": "111.111.111-11",
            "address": "Rua 1",
        },
        bad_name = {
            "name": "Arjan >-<",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate ={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data = {
            "email": "<bad data>",
            "password": "<bad data>",
        }
    )

test_data = {
        "good_data": {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
            "cpf": "11111111111",
            "address": "Rua AB 123 - Centro",
        }
    }

print("TESTE:")
try:
    user = User(**test_data["good_data"])
    print("Usuário criado!")
    print(user.model_dump_json(indent=2)) 
except Exception as e:
        print(f"Erro: {e}")

print("\nteste erro de cpf:")
try:
    bad_cpf = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "cpf": "123", 
        "address": "Rua teste 123"
     }
    User(**bad_cpf)
except Exception as e:
        print(f"Validação ok. Erro capturado:\n{e}")

main()

TESTE:
Usuário criado!
{
  "name": "Arjan",
  "email": "example@arjancodes.com",
  "role": 2,
  "cpf": "11111111111",
  "address": "Rua AB 123 - Centro"
}

teste erro de cpf:
Validação ok. Erro capturado:
1 validation error for User
cpf
  Value error, CPF is invalid, must contain exactly 11 numbers [type=value_error, input_value='123', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


In [ ]:
#Serializacao pós leitura para colocar os dados em formato json para serem lidos pela API
#Objetivo de esconder a senha para que a mesma não seja vista no arquivo final
@field_serializer("role", when_used="json")
@classmethod
def serialize_role(cls, v) -> str:
        return v.name

@model_serializer(mode="wrap", when_used="json")
def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {
                "name": self.name,
                "role": self.role.name,
                "cpf": self.cpf,
                "address": self.address,
            }
        return serializer(self)

def teste_serializacao():
    dados_exemplo = {
        "name": "Arjan",
        "email": "contato@arjancodes.com",
        "password": "Password123",
        "role": Role.Admin,
        "cpf": "11111111111",
        "address": "Rua das Flores, 123"
    }
    
    user = User(**dados_exemplo)

    print("teste de serialiacao")
    print(user)

    print("\nJSON:")
    json_resultado = user.model_dump_json(indent=2)
    print(json_resultado)

    # Teste de verificação
    if '"role": "Admin"' in json_resultado:
        print("\nO Enum Role foi convertido")
    
    #Caso a senha tenha sido excluida, havera uma mensagem falando isso
    if '"password"' not in json_resultado:
        print("Excluido a senha do arquivo final")

teste_serializacao()

teste de serialiacao
name='Arjan' email='contato@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 2> cpf='11111111111' address='Rua das Flores, 123'

JSON:
{
  "name": "Arjan",
  "email": "contato@arjancodes.com",
  "role": 2,
  "cpf": "11111111111",
  "address": "Rua das Flores, 123"
}
Excluido a senha do arquivo final
